In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

In [2]:
df = pd.read_csv("data.csv")
df.head()

,class_name,tempo,beats,chroma_stft,rmse,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,mfcc1,...,mfcc12,mfcc13,mfcc14,mfcc15,mfcc16,mfcc17,mfcc18,mfcc19,mfcc20,label
0,blues,103.359375,50,0.380260,0.248262,2116.942959,1956.611056,4196.107960,0.127272,-26.929785,...,14.336612,-13.821769,7.562789,-6.181372,0.330165,-6.829571,0.965922,-7.570825,2.918987,blues
1,blues,95.703125,44,0.306451,0.113475,1156.070496,1497.668176,2170.053545,0.058613,-233.860772,...,-2.250578,3.959198,5.322555,0.812028,-1.107202,-4.556555,-2.436490,3.316913,-0.608485,blues
2,blues,151.999081,75,0.253487,0.151571,1331.073970,1973.643437,2900.174130,0.042967,-221.802549,...,-13.037723,-12.652228,-1.821905,-7.260097,-6.660252,-14.682694,-11.719264,-11.025216,-13.387260,blues
3,blues,184.570312,91,0.269320,0.119072,1361.045467,1567.804596,2739.625101,0.069124,-207.208080,...,-0.613248,0.384877,2.605128,-5.188924,-9.527455,-9.244394,-2.848274,-1.418707,-5.932607,blues
4,blues,161.499023,74,0.391059,0.137728,1811.076084,2052.332563,3927.809582,0.075480,-145.434568,...,7.457218,-10.470444,-2.360483,-6.783623,2.671134,-4.760879,-0.949005,0.024832,-2.005315,blues


In [3]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nLabel counts:\n", df["label"].value_counts())

Shape: (1000, 30)

Columns:
 ['class_name', 'tempo', 'beats', 'chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth', 'rolloff', 'zero_crossing_rate', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20', 'label']

Missing values:
 class_name            0
tempo                 0
beats                 0
chroma_stft           0
rmse                  0
spectral_centroid     0
spectral_bandwidth    0
rolloff               0
zero_crossing_rate    0
mfcc1                 0
mfcc2                 0
mfcc3                 0
mfcc4                 0
mfcc5                 0
mfcc6                 0
mfcc7                 0
mfcc8                 0
mfcc9                 0
mfcc10                0
mfcc11                0
mfcc12                0
mfcc13                0
mfcc14                0
mfcc15                0
mfcc16                0
mfcc17               

In [4]:
feature_cols = [
    'tempo',
    'beats',
    'chroma_stft',
    'rmse',
    'spectral_centroid',
    'spectral_bandwidth',
    'rolloff',
    'zero_crossing_rate',
    'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5',
    'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10',
    'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15',
    'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20'
]

X = df[feature_cols].copy()
y = df["label"].copy()

## Encode labels

In [5]:
target_names = sorted(df["label"].unique())

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target names:", target_names)

X shape: (1000, 28)
y shape: (1000,)
Target names: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## Train-test split

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [9]:
baseline_results = []

for name, model in baseline_models.items():
    if name in ["Random Forest", "Extra Trees", "Gradient Boosting"]:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    baseline_results.append([name, acc])

baseline_df = pd.DataFrame(baseline_results, columns=["Model", "Test Accuracy"])
baseline_df = baseline_df.sort_values(by="Test Accuracy", ascending=False)
baseline_df

,Model,Test Accuracy
0,Logistic Regression,0.675
1,KNN,0.670
4,Extra Trees,0.670
2,SVM,0.660
3,Random Forest,0.650
5,Gradient Boosting,0.650


In [10]:
def evaluate_model(model, X_train_data, y_train, X_test_data, y_test, target_names, model_name="Model"):
    model.fit(X_train_data, y_train)
    y_pred = model.predict(X_test_data)

    acc = accuracy_score(y_test, y_pred)

    print(f"===== {model_name} =====")
    print("Accuracy:", round(acc, 4))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, labels=target_names, target_names=target_names))

    return acc, y_pred

In [11]:
svm_params = {
    "C": [0.1, 1, 10, 50, 100, 200],
    "kernel": ["rbf", "linear", "poly"],
    "gamma": ["scale", "auto", 0.1, 0.01, 0.001]
}

svm_grid = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

svm_grid.fit(X_train_scaled, y_train)

print("Best SVM Params:", svm_grid.best_params_)
print("Best SVM CV Score:", svm_grid.best_score_)

best_svm = svm_grid.best_estimator_

Fitting 5 folds for each of 90 candidates, totalling 450 fits
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.0s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.0s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.0s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.1s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END .....................C=0.1, gamma=scale, kernel=rbf; total time=   0.0s
[CV] END ....................C=0.1, gamma=scale, kernel=poly; total time=   0.0s
[CV] END ....................C=0.1, gamma=scale

In [12]:
svm_acc, svm_pred = evaluate_model(
    best_svm,
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    target_names=target_names,
    model_name="Tuned SVM"
)

===== Tuned SVM =====
Accuracy: 0.675

Confusion Matrix:
[[13  0  2  0  1  1  2  0  0  1]
 [ 0 17  0  0  1  1  0  0  0  1]
 [ 0  0 15  0  0  1  0  0  0  4]
 [ 0  0  1 12  0  0  1  0  2  4]
 [ 0  0  1  1 14  0  0  0  3  1]
 [ 1  2  2  0  0 14  0  0  1  0]
 [ 0  0  0  0  0  1 19  0  0  0]
 [ 0  0  2  4  0  0  0 14  0  0]
 [ 0  0  0  3  4  2  0  2  9  0]
 [ 1  0  0  6  0  2  0  0  3  8]]

Classification Report:
              precision    recall  f1-score   support

       blues       0.87      0.65      0.74        20
   classical       0.89      0.85      0.87        20
     country       0.65      0.75      0.70        20
       disco       0.46      0.60      0.52        20
      hiphop       0.70      0.70      0.70        20
        jazz       0.64      0.70      0.67        20
       metal       0.86      0.95      0.90        20
         pop       0.88      0.70      0.78        20
      reggae       0.50      0.45      0.47        20
        rock       0.42      0.40      0.41    

In [13]:
et_params = {
    "n_estimators": [200, 300, 400, 500, 700],
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

et_random = RandomizedSearchCV(
    estimator=ExtraTreesClassifier(random_state=42),
    param_distributions=et_params,
    n_iter=35,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

et_random.fit(X_train, y_train)

print("Best Extra Trees Params:", et_random.best_params_)
print("Best Extra Trees CV Score:", et_random.best_score_)

best_et = et_random.best_estimator_

Fitting 5 folds for each of 35 candidates, totalling 175 fits
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.3s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.3s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.3s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.3s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   0.3s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimators=400; total time=   0.7s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimators=400; total time=   0.7s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimato

In [14]:
et_acc, et_pred = evaluate_model(
    best_et,
    X_train, y_train,
    X_test, y_test,
    target_names=target_names,
    model_name="Tuned Extra Trees"
)

===== Tuned Extra Trees =====
Accuracy: 0.685

Confusion Matrix:
[[14  0  0  1  0  0  3  0  1  1]
 [ 0 19  1  0  0  0  0  0  0  0]
 [ 0  0 14  1  0  1  0  1  1  2]
 [ 0  0  0 10  2  0  2  1  1  4]
 [ 0  0  0  2 12  0  0  2  3  1]
 [ 0  2  3  0  0 13  0  1  1  0]
 [ 0  0  0  1  0  0 19  0  0  0]
 [ 0  0  3  2  0  0  0 15  0  0]
 [ 0  1  1  1  3  1  1  0 12  0]
 [ 0  0  0  5  1  2  0  2  1  9]]

Classification Report:
              precision    recall  f1-score   support

       blues       1.00      0.70      0.82        20
   classical       0.86      0.95      0.90        20
     country       0.64      0.70      0.67        20
       disco       0.43      0.50      0.47        20
      hiphop       0.67      0.60      0.63        20
        jazz       0.76      0.65      0.70        20
       metal       0.76      0.95      0.84        20
         pop       0.68      0.75      0.71        20
      reggae       0.60      0.60      0.60        20
        rock       0.53      0.45      

In [15]:
rf_params = {
    "n_estimators": [200, 300, 400, 500, 700],
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=rf_params,
    n_iter=35,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rf_random.fit(X_train, y_train)

print("Best RF Params:", rf_random.best_params_)
print("Best RF CV Score:", rf_random.best_score_)

best_rf = rf_random.best_estimator_

Fitting 5 folds for each of 35 candidates, totalling 175 fits
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   1.1s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   1.1s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   1.1s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   1.2s
[CV] END max_depth=40, max_features=None, min_samples_leaf=4, min_samples_split=10, n_estimators=200; total time=   1.1s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimators=400; total time=   2.3s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimators=400; total time=   2.4s
[CV] END max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=5, n_estimato

In [16]:
rf_acc, rf_pred = evaluate_model(
    best_rf,
    X_train, y_train,
    X_test, y_test,
    target_names=target_names,
    model_name="Tuned Random Forest"
)

===== Tuned Random Forest =====
Accuracy: 0.655

Confusion Matrix:
[[14  0  0  1  0  0  3  0  1  1]
 [ 0 18  1  0  0  1  0  0  0  0]
 [ 0  0 12  1  0  3  0  1  1  2]
 [ 0  0  0 12  3  0  2  0  1  2]
 [ 0  0  0  2 12  0  0  2  3  1]
 [ 0  2  4  0  0 12  0  1  1  0]
 [ 0  0  0  1  1  0 17  0  1  0]
 [ 0  0  3  2  0  0  0 15  0  0]
 [ 0  0  0  1  2  2  1  1 13  0]
 [ 1  0  1  5  1  1  0  3  2  6]]

Classification Report:
              precision    recall  f1-score   support

       blues       0.93      0.70      0.80        20
   classical       0.90      0.90      0.90        20
     country       0.57      0.60      0.59        20
       disco       0.48      0.60      0.53        20
      hiphop       0.63      0.60      0.62        20
        jazz       0.63      0.60      0.62        20
       metal       0.74      0.85      0.79        20
         pop       0.65      0.75      0.70        20
      reggae       0.57      0.65      0.60        20
        rock       0.50      0.30    

In [17]:
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11, 13, 15, 17, 21],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"]
}

knn_grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

knn_grid.fit(X_train_scaled, y_train)

print("Best KNN Params:", knn_grid.best_params_)
print("Best KNN CV Score:", knn_grid.best_score_)

best_knn = knn_grid.best_estimator_

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ..metric=euclidean, n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=5, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=5, weights=uniform; total time=   0.0s
[CV] END ...metric=euclidean, n_neighbors=5, we

In [18]:
knn_acc, knn_pred = evaluate_model(
    best_knn,
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    target_names=target_names,
    model_name="Tuned KNN"
)

===== Tuned KNN =====
Accuracy: 0.64

Confusion Matrix:
[[13  0  1  2  0  1  2  0  1  0]
 [ 0 18  1  0  0  0  0  0  0  1]
 [ 0  0 14  2  0  1  0  2  0  1]
 [ 0  0  0 11  3  0  1  2  1  2]
 [ 0  0  2  1 11  0  0  2  2  2]
 [ 0  3  3  0  0 10  0  4  0  0]
 [ 1  0  1  0  1  0 15  0  0  2]
 [ 0  0  1  3  0  0  0 15  0  1]
 [ 0  1  2  0  4  0  0  1 12  0]
 [ 0  0  0  6  0  2  0  2  1  9]]

Classification Report:
              precision    recall  f1-score   support

       blues       0.93      0.65      0.76        20
   classical       0.82      0.90      0.86        20
     country       0.56      0.70      0.62        20
       disco       0.44      0.55      0.49        20
      hiphop       0.58      0.55      0.56        20
        jazz       0.71      0.50      0.59        20
       metal       0.83      0.75      0.79        20
         pop       0.54      0.75      0.62        20
      reggae       0.71      0.60      0.65        20
        rock       0.50      0.45      0.47     

In [19]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

selector = SelectKBest(score_func=f_classif, k=20)

X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)

scaler_kbest = StandardScaler()
X_train_kbest_scaled = scaler_kbest.fit_transform(X_train_kbest)
X_test_kbest_scaled = scaler_kbest.transform(X_test_kbest)

# Use the tuned SVM settings
svm_fs = SVC(
    C=best_svm.C,
    kernel=best_svm.kernel,
    gamma=best_svm.gamma
)

svm_fs.fit(X_train_kbest_scaled, y_train)
svm_fs_pred = svm_fs.predict(X_test_kbest_scaled)
svm_fs_acc = accuracy_score(y_test, svm_fs_pred)

print("Tuned SVM + SelectKBest Accuracy:", svm_fs_acc)

Tuned SVM + SelectKBest Accuracy: 0.705


In [20]:
final_results = pd.DataFrame({
    "Model": [
        "Tuned SVM",
        "Tuned Extra Trees",
        "Tuned Random Forest",
        "Tuned KNN",
        "Tuned SVM + SelectKBest"
    ],
    "Test Accuracy": [svm_acc, et_acc, rf_acc, knn_acc, svm_fs_acc]
}).sort_values(by="Test Accuracy", ascending=False)

final_results

,Model,Test Accuracy
4,Tuned SVM + SelectKBest,0.705
1,Tuned Extra Trees,0.685
0,Tuned SVM,0.675
2,Tuned Random Forest,0.655
3,Tuned KNN,0.640


In [21]:
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask]
print("Selected features:")
print(selected_features.tolist())

Selected features:
['chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth', 'rolloff', 'zero_crossing_rate', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10', 'mfcc12', 'mfcc13', 'mfcc15', 'mfcc17']


In [22]:
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_cv_scores = cross_val_score(
    best_svm,
    X_scaled,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("SVM CV scores:", svm_cv_scores)
print("SVM Mean CV accuracy:", svm_cv_scores.mean())

SVM CV scores: [0.685 0.675 0.65  0.67  0.705]
SVM Mean CV accuracy: 0.677


In [23]:
et_cv_scores = cross_val_score(
    best_et,
    X,
    y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("ET CV scores:", et_cv_scores)
print("ET Mean CV accuracy:", et_cv_scores.mean())

ET CV scores: [0.635 0.7   0.67  0.715 0.69 ]
ET Mean CV accuracy: 0.6819999999999999


In [24]:
model_dict = {
    "Tuned SVM": best_svm,
    "Tuned Extra Trees": best_et,
    "Tuned Random Forest": best_rf,
    "Tuned KNN": best_knn,
    "Tuned SVM + SelectKBest": svm_fs
}

best_model_name = final_results.iloc[0]["Model"]
best_model = model_dict[best_model_name]

print("Best model:", best_model_name)
print(best_model)

Best model: Tuned SVM + SelectKBest
SVC(C=10)


In [25]:
if best_model_name == "Tuned SVM + SelectKBest":
    model_package = {
        "model_name": best_model_name,
        "model": best_model,
        "selector": selector,
        "scaler": scaler_kbest,
        "feature_columns": feature_cols
    }
else:
    model_package = {
        "model_name": best_model_name,
        "model": best_model,
        "scaler": scaler,
        "feature_columns": feature_cols
    }

MODEL_SAVE_PATH = "/Users/milind/Downloads/Music_Genre_Classification/best_music_genre_model.pkl"

with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(model_package, f)

print("Saved model to:", MODEL_SAVE_PATH)

Saved model to: /Users/milind/Downloads/Music_Genre_Classification/best_music_genre_model.pkl


In [26]:
with open("/Users/milind/Downloads/Music_Genre_Classification/best_music_genre_model.pkl", "rb") as f:
    saved_package = pickle.load(f)

saved_package.keys()

dict_keys(['model_name', 'model', 'selector', 'scaler', 'feature_columns'])

In [27]:
from Metadata import getmetadata
import pandas as pd

def predict_genre(file_path, model_package):
    model = model_package["model"]
    model_name = model_package["model_name"]
    feature_columns = model_package["feature_columns"]

    meta = getmetadata(file_path)

    # Build DataFrame in exact training feature order
    sample_df = pd.DataFrame([[meta[col] for col in feature_columns]], columns=feature_columns)

    if model_name == "Tuned SVM + SelectKBest":
        selector = model_package["selector"]
        scaler = model_package["scaler"]

        sample_selected = selector.transform(sample_df)
        sample_scaled = scaler.transform(sample_selected)
        pred = model.predict(sample_scaled)[0]
        return pred

    elif model_name in ["Tuned SVM", "Tuned KNN"]:
        scaler = model_package["scaler"]
        sample_scaled = scaler.transform(sample_df)
        pred = model.predict(sample_scaled)[0]
        return pred

    else:
        pred = model.predict(sample_df)[0]
        return pred

In [28]:
import importlib
import Metadata

importlib.reload(Metadata)
from Metadata import getmetadata

In [29]:
meta = getmetadata("/Users/milind/Downloads/Music_Genre_Classification/3.wav")
print(meta.keys())
print("beats" in meta)

dict_keys(['tempo', 'beats', 'chroma_stft', 'rmse', 'spectral_centroid', 'spectral_bandwidth', 'rolloff', 'zero_crossing_rate', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7', 'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14', 'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20'])
True


In [30]:
predicted_genre = predict_genre("/Users/milind/Downloads/Music_Genre_Classification/3.wav", saved_package)
print("Predicted Genre:", predicted_genre)

Predicted Genre: classical
